# 02 — Análise Exploratória de Dados (EDA)

Exploramos os dados coletados antes de construir o grafo.
Execute `01_coleta.ipynb` primeiro para gerar o arquivo `data/raw/works_ufms_cs.json`.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from pathlib import Path

from src.fetch import carregar_raw

sns.set_theme(style='whitegrid', palette='muted')
FIGURES = Path('../reports/figures')
FIGURES.mkdir(parents=True, exist_ok=True)

works = carregar_raw('works_ufms_cs')
print(f'{len(works)} papers carregados')

## 1. Distribuição Temporal

In [ ]:
anos = [w['publication_year'] for w in works if w.get('publication_year')]

fig, ax = plt.subplots(figsize=(11, 4))
pd.Series(anos).value_counts().sort_index().plot(kind='bar', color='#01696f', ax=ax)
ax.set_title('Publicações por Ano', fontsize=14)
ax.set_xlabel('Ano')
ax.set_ylabel('Número de papers')
plt.tight_layout()
plt.savefig(FIGURES / 'publicacoes_por_ano.png', dpi=150)
plt.show()

## 2. Top 20 Conceitos

In [ ]:
all_concepts = [c['display_name'] for w in works for c in w.get('concepts', [])[:3]]
top_concepts = pd.Series(Counter(all_concepts)).nlargest(20)

fig, ax = plt.subplots(figsize=(10, 8))
top_concepts.sort_values().plot(kind='barh', color='#01696f', ax=ax)
ax.set_title('Top 20 Conceitos', fontsize=14)
plt.tight_layout()
plt.savefig(FIGURES / 'top_concepts.png', dpi=150)
plt.show()

## 3. Top 20 Autores

In [ ]:
autores = [
    a['author']['display_name']
    for w in works
    for a in w.get('authorships', [])
    if a.get('author') and a['author'].get('display_name')
]

fig, ax = plt.subplots(figsize=(10, 8))
pd.Series(Counter(autores)).nlargest(20).sort_values().plot(kind='barh', color='#01696f', ax=ax)
ax.set_title('Top 20 Autores por Número de Papers', fontsize=14)
plt.tight_layout()
plt.savefig(FIGURES / 'top_autores.png', dpi=150)
plt.show()

## 4. Open Access

In [ ]:
oa_status = [
    w.get('open_access', {}).get('oa_status', 'unknown')
    for w in works
]

fig, ax = plt.subplots(figsize=(7, 4))
pd.Series(oa_status).value_counts().plot(kind='bar', color='#01696f', ax=ax)
ax.set_title('Distribuição de Status Open Access', fontsize=14)
ax.set_xlabel('')
plt.tight_layout()
plt.savefig(FIGURES / 'open_access.png', dpi=150)
plt.show()

## 5. Citações — Top 10 Papers Mais Citados

In [ ]:
df = pd.DataFrame([{
    'title': w.get('title', 'N/A')[:60],
    'year': w.get('publication_year'),
    'cited_by_count': w.get('cited_by_count', 0)
} for w in works])

df.sort_values('cited_by_count', ascending=False).head(10)